In [ ]:
from __main__ import LatticeSimulation
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path



Lattice simulation notebook. this notebook contains code to simulate various lattice models usingmethods. We begin by by simulating a simple 2D lattice and plotting the DOS(Density of States), for Dirac(relativistic)  and Schrodinger Hamiltonians.

In [ ]:
#simulation parameters:
Lx,Ly=30,30 #lattice size
t=1.0 #hopping parameter
m=0.0 #mass term for Dirac Hamiltonian
bcs = ['pbc', 'obc']
results = {}
for bc in bcs:
    print(f"Running simulation for {bc.upper()}...")
    sim = LatticeSimulation(Lx, Ly, boundary_condition=bc)
    results[bc] = {}

    for mass in mass_values:
        print(f"  Mass = {mass}")
            # Dirac
        H_dir = sim.get_dirac_hamiltonian(t=t_hop, m=mass)
        H_sch=sim.get_schrodinger_hamiltonian(t=t_hop)
        energies = sim.get_spectrum(H_dir)#eigenvalues of the hamiltonian
        real_dos=(1/(8*np.pi*t_hop**2))*np.abs(energies)#theoretical density of states for 2D Dirac
        #print(real_dos)
        window = 2.0 * abs(mass)#keep energies within this window -2m<e<2m
        #window=0.5
            filtered_energies = energies[(energies >= -window) & (energies <= window)]
            results[bc][mass] = filtered_energies
            #results[bc][mass] = energies
    # --- Plotting Dirac Results for All Masses ---
    print("Plotting Dirac results for all masses...")
    fig, (ax_pbc, ax_obc) = plt.subplots(1, 2, figsize=(16, 8))
    colors = ['red', 'blue', 'green', 'orange', 'purple', 'brown']
    # PBC plot
    for i, mass in enumerate(mass_values):
        n,edges,patches=ax_pbc.hist(results['pbc'][mass], bins=30, color=colors[i],
                   alpha=0.6, label=f'm={mass}', edgecolor='black', linewidth=0.5)

    ax_pbc.set_title(f"Dirac PBC -m={mass} ,Lattice sites: {Lx}x{Ly}", fontsize=14)
    dE=edges[1]-edges[0]#bin width
    ax_pbc.set_ylabel("DOS (Count)", fontsize=12)
    ax_pbc.set_xlabel("Energy (E)", fontsize=12)
    ax_pbc.legend()
    ax_pbc.grid(True, alpha=0.3)
    real_dos=real_dos*dE*Lx*Ly#scale theoretical dos by bin width
    ax_pbc.plot(energies, real_dos, color='black', label='Theoretical DOS')
    ax_pbc.legend()
    # OBC plot
    for i, mass in enumerate(mass_values):
        ax_obc.hist(results['obc'][mass], bins=30, color=colors[i],
                   alpha=0.6, label=f'm={mass}', edgecolor='black', linewidth=0.5)

    ax_obc.set_title(f"Dirac OBC - Multiple Masses\nLattice: {Lx}x{Ly}", fontsize=14)
    ax_obc.set_ylabel("DOS (Count)", fontsize=12)
    ax_obc.set_xlabel("Energy (E)", fontsize=12)
    ax_obc.plot(energies, real_dos, color='black', label='Theoretical DOS')
    ax_obc.legend()
    ax_obc.grid(True, alpha=0.3)
    out_dir = Path(__file__).resolve().parent / "plots"
    out_dir.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    plt.savefig(out_dir / f"dirac_real_vs_numerical.jpg", dpi=300)
    plt.show()

